# 05 — Extended and Distributed Lag Analysis

This notebook documents the previously completed extended-lag analyses.

- **Stage 15:** historical terminal outputs are preserved and loaded from the frozen reference.
- **Stage 16:** the distributed-lag workflow is reproducibly regenerated from code.

The migration objective is to organize the completed research in Jupyter without unnecessarily re-engineering analyses that were already completed.

In [ ]:
from pathlib import Path
import subprocess
import sys

import pandas as pd
from IPython.display import display

from groundwater_insar.config import load_config
from groundwater_insar.stage16_distributed_lag import reconstruct_stage16

cfg = load_config()

repo = Path(cfg["paths"]["repository_root"])

def git(*args):
    return subprocess.check_output(
        ["git", "-C", str(repo), *args],
        text=True
    ).strip()

print("Python     :", sys.version.split()[0])
print("Git commit :", git("rev-parse", "HEAD"))
print("Git dirty  :", bool(git("status", "--porcelain")))

## 1. Stage 15 — Historical extended CCF

Stage 15 extended the cross-correlation analysis to lags from **0 to 36 months** for the 37 eligible piezometers.

Two versions were evaluated:

- `RAW`
- `SEASONAL_ADJUSTED`

The frozen terminal outputs are retained as the authoritative historical record for this stage.

During the Jupyter migration, Stage 15 was **not re-engineered further**, because the objective of this migration is organization and reproducibility of the completed workflow rather than reconstructing every intermediate terminal implementation from scratch.

In [ ]:
reference = Path(
    "/mnt/d/ALPRIFT_DSS_Research/"
    "references/Groundwater_InSAR_Memory/"
    "v1_terminal_baseline/csv"
)

ccf15 = pd.read_csv(
    reference / "15_CCF_Lag0_36_ByWell.csv"
)

best15 = pd.read_csv(
    reference / "15_BestLag0_36_ByWell.csv"
)

windows15 = pd.read_csv(
    reference / "15_BestLag_By_PhysicalWindow.csv"
)

summary15 = pd.read_csv(
    reference / "15_CCF_Lag0_36_Summary.csv"
)

print("CCF rows        :", len(ccf15))
print("Best-lag rows   :", len(best15))
print("Window rows     :", len(windows15))
print("Summary rows    :", len(summary15))
print("Wells           :", ccf15["Piezometer"].nunique())

display(best15.head(10))

### Interpretation of Stage 15

The extended CCF analysis did not identify a consistent basin-wide fixed lag.

Therefore, a single lag value should not be interpreted as the universal groundwater–subsidence response time of the Shabestar plain.

## 2. Stage 16 — Binned distributed-lag analysis

Groundwater decline is represented using four fixed lag windows:

- 0–6 months
- 7–12 months
- 13–24 months
- 25–36 months

Stage 16 is regenerated directly from the version-controlled Python implementation.

In [ ]:
reconstruct_stage16(cfg)

stage16 = (
    Path(cfg["paths"]["regenerated_root"])
    / "stage16"
)

panel16 = pd.read_csv(
    stage16 / "16_Binned_DistributedLag_Panel.csv"
)

corr16 = pd.read_csv(
    stage16 / "16_Binned_DistributedLag_Correlations_ByWell.csv"
)

betas16 = pd.read_csv(
    stage16 / "16_Binned_DistributedLag_Betas_ByWell.csv"
)

print("Panel rows       :", len(panel16))
print("Wells            :", panel16["Piezometer"].nunique())
print("Correlation rows :", len(corr16))
print("Beta rows        :", len(betas16))

display(betas16.head(10))

In [ ]:
summary_path = (
    stage16 / "16_Binned_DistributedLag_Summary.csv"
)

print(summary_path.read_text(encoding="utf-8"))

## 3. Stage 16 frozen-baseline regression

The regenerated distributed-lag outputs are compared against the frozen terminal baseline.

In [ ]:
result = subprocess.run(
    [
        sys.executable,
        "-m",
        "groundwater_insar.regression",
        "--candidate-dir",
        str(stage16),
        "--stage",
        "16",
        "--report-name",
        "16_Stage16_Distributed_Lag_Regression.csv",
    ],
    text=True,
    capture_output=True,
)

print(result.stdout)

if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("Stage 16 regression failed.")

print("STAGE 16 REGRESSION: PASS")

## 4. Scientific checkpoint

The extended CCF and distributed-lag analyses do not support a simple, spatially uniform fixed lag between groundwater decline and land subsidence.

The pooled distributed-lag explanatory power is very low. Consequently, later analyses focus on cumulative behaviour, recovery episodes and residual subsidence rather than assigning one universal lag to the entire aquifer.

In [ ]:
checkpoint = pd.DataFrame({
    "Stage": [
        "15 — Extended CCF",
        "16 — Distributed lag",
    ],
    "Migration status": [
        "Historical frozen outputs documented",
        "Reproducibly regenerated — PASS",
    ],
})

display(checkpoint)

print("NOTEBOOK 05: PASS")